In [1]:
# Importing necessary libraries
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


In [2]:
# Loading dataset
data = pd.read_csv('churn_prediction(in).csv')

In [3]:
data.head()

,customer_id,vintage,age,gender,dependents,occupation,city,customer_nw_category,branch_code,days_since_last_transaction,...,previous_month_end_balance,average_monthly_balance_prevQ,average_monthly_balance_prevQ2,current_month_credit,previous_month_credit,current_month_debit,previous_month_debit,current_month_balance,previous_month_balance,churn
0,1,3135,66,Male,0.0,self_employed,187.0,2,755,224.0,...,1458.71,1458.71,1449.07,0.20,0.20,0.20,0.20,1458.71,1458.71,0
1,2,310,35,Male,0.0,self_employed,NaN,2,3214,60.0,...,8704.66,7799.26,12419.41,0.56,0.56,5486.27,100.56,6496.78,8787.61,0
2,4,2356,31,Male,0.0,salaried,146.0,2,41,NaN,...,5815.29,4910.17,2815.94,0.61,0.61,6046.73,259.23,5006.28,5070.14,0
3,5,478,90,NaN,NaN,self_employed,1020.0,2,582,147.0,...,2291.91,2084.54,1006.54,0.47,0.47,0.47,2143.33,2291.91,1669.79,1
4,6,2531,42,Male,2.0,self_employed,1494.0,3,388,58.0,...,1401.72,1643.31,1871.12,0.33,714.61,588.62,1538.06,1157.15,1677.16,1


In [4]:
# Summary statistics
print(data.describe())

        customer_id       vintage           age    dependents          city  \
count  28382.000000  28382.000000  28382.000000  25919.000000  27579.000000   
mean   15143.508667   2364.336446     48.208336      0.347236    796.109576   
std     8746.454456   1610.124506     17.807163      0.997661    432.872102   
min        1.000000    180.000000      1.000000      0.000000      0.000000   
25%     7557.250000   1121.000000     36.000000      0.000000    409.000000   
50%    15150.500000   2018.000000     46.000000      0.000000    834.000000   
75%    22706.750000   3176.000000     60.000000      0.000000   1096.000000   
max    30301.000000  12899.000000     90.000000     52.000000   1649.000000   

       customer_nw_category   branch_code  days_since_last_transaction  \
count          28382.000000  28382.000000                 25159.000000   
mean               2.225530    925.975019                    69.997814   
std                0.660443    937.799129                    86.34

In [5]:
# Checking for null values.
print(data.isnull().sum())

customer_id                          0
vintage                              0
age                                  0
gender                             525
dependents                        2463
occupation                          80
city                               803
customer_nw_category                 0
branch_code                          0
days_since_last_transaction       3223
current_balance                      0
previous_month_end_balance           0
average_monthly_balance_prevQ        0
average_monthly_balance_prevQ2       0
current_month_credit                 0
previous_month_credit                0
current_month_debit                  0
previous_month_debit                 0
current_month_balance                0
previous_month_balance               0
churn                                0
dtype: int64


In [6]:
# Dropping rows with null values.
newdata = data.dropna()

In [7]:
# Checking for null values.
print(newdata.isnull().sum())

customer_id                       0
vintage                           0
age                               0
gender                            0
dependents                        0
occupation                        0
city                              0
customer_nw_category              0
branch_code                       0
days_since_last_transaction       0
current_balance                   0
previous_month_end_balance        0
average_monthly_balance_prevQ     0
average_monthly_balance_prevQ2    0
current_month_credit              0
previous_month_credit             0
current_month_debit               0
previous_month_debit              0
current_month_balance             0
previous_month_balance            0
churn                             0
dtype: int64


In [8]:
# Checking variable data types.
newdata.dtypes

customer_id                         int64
vintage                             int64
age                                 int64
gender                             object
dependents                        float64
occupation                         object
city                              float64
customer_nw_category                int64
branch_code                         int64
days_since_last_transaction       float64
current_balance                   float64
previous_month_end_balance        float64
average_monthly_balance_prevQ     float64
average_monthly_balance_prevQ2    float64
current_month_credit              float64
previous_month_credit             float64
current_month_debit               float64
previous_month_debit              float64
current_month_balance             float64
previous_month_balance            float64
churn                               int64
dtype: object

In [9]:
# Importing preprocesing and label encoder.
# Encode labels in column 'gender'. 
from sklearn import preprocessing  
label_encoder = preprocessing.LabelEncoder() 
newdata['gender']= label_encoder.fit_transform(newdata['gender']) 
newdata['gender'].unique()

C:\Users\cse21-102\AppData\Local\Temp\ipykernel_8524\380654594.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  newdata['gender']= label_encoder.fit_transform(newdata['gender'])


array([1, 0])

In [10]:
# Creating one-hot encoded columns for each unique value in the occupation column.
# The new columns will have a prefix of 'occupation_' followed by the unique value from the occupation column.
# The resulting one-hot encoded columns are then concatenated to the original 'df' dataframe along the columns axis (axis=1).
newdata = pd.concat([newdata, pd.get_dummies(newdata['occupation'], prefix=str('occupation'), prefix_sep='_')], axis=1)
newdata.head()

,customer_id,vintage,age,gender,dependents,occupation,city,customer_nw_category,branch_code,days_since_last_transaction,...,current_month_debit,previous_month_debit,current_month_balance,previous_month_balance,churn,occupation_company,occupation_retired,occupation_salaried,occupation_self_employed,occupation_student
0,1,3135,66,1,0.0,self_employed,187.0,2,755,224.0,...,0.20,0.20,1458.71,1458.71,0,0,0,0,1,0
4,6,2531,42,1,2.0,self_employed,1494.0,3,388,58.0,...,588.62,1538.06,1157.15,1677.16,1,0,0,0,1,0
5,7,263,42,0,0.0,self_employed,1096.0,2,1666,60.0,...,857.50,286.07,15719.44,15349.75,0,0,0,0,1,0
6,8,5922,72,1,0.0,retired,1020.0,1,1,98.0,...,1299.64,439.26,7076.06,7755.98,0,0,1,0,0,0
7,9,1145,46,1,0.0,self_employed,623.0,2,317,172.0,...,443.13,5688.44,8563.84,5317.04,0,0,0,0,1,0


In [11]:
# Checking variable data types.
newdata.dtypes

customer_id                         int64
vintage                             int64
age                                 int64
gender                              int32
dependents                        float64
occupation                         object
city                              float64
customer_nw_category                int64
branch_code                         int64
days_since_last_transaction       float64
current_balance                   float64
previous_month_end_balance        float64
average_monthly_balance_prevQ     float64
average_monthly_balance_prevQ2    float64
current_month_credit              float64
previous_month_credit             float64
current_month_debit               float64
previous_month_debit              float64
current_month_balance             float64
previous_month_balance            float64
churn                               int64
occupation_company                  uint8
occupation_retired                  uint8
occupation_salaried               

In [13]:
# Dropping occupation colum
newdata = newdata.drop('occupation', axis=1)

In [14]:
# Split the data into features (X) and target (y).
X = newdata.drop('churn', axis=1)
y = newdata['churn']

In [17]:
# Importing StandardScaler.
from sklearn.preprocessing import StandardScaler 

# Split the data into training and test sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features using StandardScaler.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
# Importing Logistic Regression 
from sklearn.linear_model import LogisticRegression

# LogisticRegression
lr = LogisticRegression(random_state=0)
lr.fit(X_train, y_train)

# Prediction
y_pred = lr.predict(X_test)

In [20]:
# Importing accuracy score
from sklearn.metrics import accuracy_score

# Calculting accuracy
acc = accuracy_score(y_test, y_pred)
print("Logistic Regression model accuracy (in %):", acc*100)

Logistic Regression model accuracy (in %): 81.28681468056185
